# 06. Обучение финальной production-модели

Этот ноутбук подробно разбирает обучение финальной модели XGBoost. В отличие от `scripts/train_final_model.py`, ноутбук предназначен для объяснения процесса. Он не сохраняет модель и не перезаписывает `artifacts/`.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:,.2f}".format)

import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_squared_log_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

from ml_pipeline.feature_engineering import MovieFeatureEngineer

DATASET_PATH = PROJECT_ROOT / "data" / "output.csv"
df = pd.read_csv(DATASET_PATH)
df.shape

## Данные и target

Финальная модель прогнозирует `gross`. Во время обучения target преобразуется через `log1p`, но пользователь и API получают обычную выручку благодаря `inverse_func=np.expm1` внутри `TransformedTargetRegressor`.

In [ ]:
TARGET_COLUMN = "gross"
MODEL_INPUT_COLUMNS = [
    "rating", "genre", "year", "score", "votes", "director", "writer",
    "star", "country", "budget", "company", "runtime",
]

train_df = df.dropna(subset=["gross", "budget"]).copy()
train_df["gross"] = pd.to_numeric(train_df["gross"], errors="coerce")
train_df["budget"] = pd.to_numeric(train_df["budget"], errors="coerce")
train_df = train_df.dropna(subset=["gross", "budget"])
train_df = train_df[(train_df["gross"] > 0) & (train_df["budget"] > 0)].copy()

X = train_df[MODEL_INPUT_COLUMNS]
y = train_df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, X_test.shape

## Сборка sklearn Pipeline

Pipeline состоит из трёх уровней:

1. `MovieFeatureEngineer` создаёт дополнительные признаки.
2. `ColumnTransformer` обрабатывает категориальные и числовые признаки.
3. `TransformedTargetRegressor` обучает XGBoost на `log1p(gross)` и возвращает прогноз через `expm1`.

In [ ]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", min_frequency=5)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore")

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", make_one_hot_encoder()),
])

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
])

preprocessor = ColumnTransformer([
    ("categorical", categorical_pipeline, MovieFeatureEngineer.categorical_features),
    ("numeric", numeric_pipeline, MovieFeatureEngineer.numeric_features),
])

xgb_regressor = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective="reg:squarederror",
)

target_regressor = TransformedTargetRegressor(
    regressor=xgb_regressor,
    func=np.log1p,
    inverse_func=np.expm1,
    check_inverse=False,
)

pipeline = Pipeline([
    ("feature_engineering", MovieFeatureEngineer()),
    ("preprocessor", preprocessor),
    ("model", target_regressor),
])

pipeline

## Обучение и оценка

Метрики считаются на обычной выручке `gross`, чтобы их можно было интерпретировать в бизнес-единицах. Дополнительно считается `log_r2`, так как модель обучается в логарифмическом пространстве.

In [ ]:
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
y_pred = np.clip(y_pred, 0, None)

metrics = {
    "mae": mean_absolute_error(y_test, y_pred),
    "rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
    "r2": r2_score(y_test, y_pred),
    "msle": mean_squared_log_error(y_test, y_pred),
    "log_r2": r2_score(np.log1p(y_test), np.log1p(y_pred)),
}

pd.Series(metrics, name="value").to_frame()

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_pred, alpha=0.45, color="#3c73ff")
limit = max(y_test.max(), y_pred.max())
plt.plot([0, limit], [0, limit], linestyle="--", color="#ff8a5b")
plt.title("Фактическая и прогнозируемая касса")
plt.xlabel("Фактическая касса, USD")
plt.ylabel("Прогноз кассы, USD")
plt.tight_layout()
plt.show()

## Проверка, что predict возвращает обычную выручку

Если бы pipeline возвращал `log_gross`, значения были бы порядка 10–25. В production-варианте прогноз должен быть в долларах США.

In [ ]:
example_prediction = float(pipeline.predict(X_test.head(1))[0])
print("Пример прогноза:", f"${example_prediction:,.0f}")
print("Это обычная выручка, а не log_gross:", example_prediction > 1_000)

## Вывод

Ноутбук показывает учебную версию обучения. Для реального экспорта используется `scripts/train_final_model.py`, который сохраняет `movie_revenue_pipeline.joblib`, `metrics.json`, `model_info.json`, `feature_schema.json` и `sample_input.json`.